# Delete job: purges deleted operations-app jobs, and their target tables if asked

The operations app has no DELETE privilege on the Lakebase tables. When an ingestion job is deleted in the app, it is
marked deleted and kept for the retention period (`deleted_job_retention_days`), so a Restore can bring it back. This
job runs as the deployer (the state database's owner) and purges it (`delete_job.py`). One mode per run:

- **A job's id (`schedule_id`):** the app's Delete now. In one transaction, under each group's run lock, the groups'
  control rows and their checkpoint, reseed_queue, database_check and check_work rows. Groups another live job holds
  stay. A run of any group refuses the lot, and nothing is deleted. The job is then marked `delete_finished`.
- **Expired (no id, no drafts; the daily run):** every deleted job whose retention has run out is purged the same way,
  each in its own transaction. A job whose group is loading waits for the next daily run. A job's target tables are
  dropped when it asked for them.
- **Drafts (`purge_drafts`):** the tables the app read for a choice no job took (discarded, or not asked about for
  `draft_ttl_s`) are purged: their discovery and discovered_table rows, in one transaction.
- **One table (`control_ids`, with `schedule_id`):** a live job's table removed in the app. Its control row, checkpoint
  and reseed_queue rows go, under its group's run lock; its target table and the job's other tables stay.
- **Kept, always:** run history (ingest_run, ingest_event) and the change log (control_change).
- **Target tables:** dropped after the commit, only the ones only the purged groups write. A table another group's
  rows also write is kept and named. A schema is never dropped.

In [ ]:
import sys, os, json

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)

from state_store import StateStore
import delete_job


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name).strip()


def flag(name, default="false"):
    return widget(name, default).lower() in ("1", "true", "yes")


schedule_id = widget("schedule_id")
drop_targets = flag("drop_targets")
db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
purge_drafts = flag("purge_drafts")
draft_ttl_s = int(widget("draft_ttl_s", str(delete_job.DRAFT_TTL_S)) or delete_job.DRAFT_TTL_S)
control_ids = [int(i) for i in widget("control_ids").replace(" ", "").split(",") if i]
purge_expired = flag("purge_expired", "true")
retention_days = int(widget("retention_days", "7") or 7)
if schedule_id.isdigit():
    mode = "table" if control_ids else "job"
elif purge_drafts:
    mode = "drafts"
else:
    assert purge_expired, "give a schedule_id (a job's Delete now), purge_drafts, or purge_expired (the daily run)"
    mode = "expired"
assert not control_ids or schedule_id.isdigit(), "control_ids need schedule_id: the ingestion job the tables belong to"

In [ ]:
import contextlib
from concurrent.futures import ThreadPoolExecutor


def drop_tables(tables):
    """Drops the tables, several at a time (a job of many thousands would take hours one by one). Returns {table:
    error} for those that failed; the rest are gone."""
    def drop(table):
        try:
            spark.sql(delete_job.drop_sql(table))
            return table, None
        except Exception as e:
            return table, str(e)[:300]

    failed = {}
    with ThreadPoolExecutor(max_workers=16) as pool:
        for table, error in pool.map(drop, tables):
            if error:
                failed[table] = error
    return failed


store = StateStore.connect_lakebase(db_instance, db_name)


@contextlib.contextmanager
def transaction():
    with store.connection.cursor() as cur, cur.connection.transaction():
        yield cur


try:
    if mode == "drafts":
        with transaction() as cur:
            purged = delete_job.purge_drafts(cur, draft_ttl_s)
    elif mode == "expired":
        purged = delete_job.purge_expired(transaction, retention_days, drop_tables)
    elif mode == "table":
        with transaction() as cur:
            removed = delete_job.delete_table_rows(cur, int(schedule_id), control_ids)
    else:
        with transaction() as cur:
            result = delete_job.delete_rows(cur, int(schedule_id))
finally:
    store.close()

In [ ]:
if mode == "drafts":
    print(f"purged {purged['drafts']} draft(s): {purged['databases']} discovery row(s), {purged['tables']} table(s)")
    dbutils.notebook.exit(json.dumps({"purged": purged}))

if mode == "expired":
    for entry in purged["purged"]:
        print(f"purged ingestion job {entry['job']}: {entry['groups']} group(s), {entry['tables']} table row(s), "
              f"{entry['dropped']} target table(s) dropped, {entry['kept']} kept")
    for entry in purged["waiting"]:
        print(f"waiting, not purged: {entry['job']}: {entry['reason']}")
    print(f"{purged['listed']} deleted job(s) past the {retention_days}-day retention: {len(purged['purged'])} "
          f"purged, {len(purged['waiting'])} waiting")
    for table, error in purged["failed"].items():
        print(f"not dropped: {table}: {error}")
    if purged["failed"]:
        raise RuntimeError(delete_job.failure_message(purged["failed"]))
    dbutils.notebook.exit(json.dumps(purged))

if mode == "table":
    print(f"removed {len(removed['control_ids'])} table(s) from ingestion job {removed['job']} ({schedule_id}): "
          f"deleted {json.dumps(removed['deleted'])}; {removed['databases']} database(s) recounted")
    if removed["missing"]:
        print(f"already gone: control ids {removed['missing']}")
    if removed["targets"]["kept"]:
        print("target table(s) kept: " + ", ".join(removed["targets"]["kept"]))
    dbutils.notebook.exit(json.dumps({"removed": removed}))

print(f"ingestion job {result['job']} ({schedule_id}): {len(result['groups'])} group(s) "
      f"{', '.join(result['groups']) or '-'}; deleted {json.dumps(result['deleted'])}")
kept = result["targets"]["kept"]
if kept:
    print(f"{len(kept)} target table(s) kept, since another group's rows also write them: "
          + ", ".join(kept[:delete_job.LISTED]) + (" and more" if len(kept) > delete_job.LISTED else ""))
to_drop = result["targets"]["drop"] if drop_targets else []
failed = drop_tables(to_drop) if to_drop else {}
dropped = [table for table in to_drop if table not in failed]
if drop_targets:
    print(f"dropped {len(dropped)} target table(s)" + (f", {len(failed)} failed" if failed else ""))
else:
    print(f"target tables kept ({len(result['targets']['drop'])} only this job wrote)")
summary = delete_job.summary(result, dropped, failed)
print(json.dumps(summary, indent=2))
if failed:
    for table, error in failed.items():
        print(f"not dropped: {table}: {error}")
    raise RuntimeError(delete_job.failure_message(failed))
dbutils.notebook.exit(json.dumps(summary))